In [1]:
import pandas as pd

In [5]:
df = pd.read_csv('labels.csv')
for row in df.rolling(1):
    print(row['audio_path'][0])

300_AUDIO.wav


In [ ]:
str(df['participant_id'])}_P', str(path['audio_path'])

# Data cleaning

### first we clean audio files to remove background noises and enhance speaker voices

we can test to see if this makes things better

In [ ]:
import os
import subprocess

INPUT_FILE = r"/Users/gurusai/Desktop/DAIC_Raw/300_P/300_AUDIO.wav"
OUTPUT_DIR = "data"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Step 1: Demucs
print("Running Demucs...")
subprocess.run([
    "demucs",
    "--two-stems=vocals",
    INPUT_FILE
])

base = os.path.splitext(os.path.basename(INPUT_FILE))[0]
demucs_out = f"separated/htdemucs/{base}/vocals.wav"

# Step 2: FFmpeg cleaning
output_path = os.path.join(OUTPUT_DIR, "cleanedv2.wav")

print("Applying FFmpeg filters...")

subprocess.run([
    "ffmpeg",
    "-i", demucs_out,
    "-af",
    # Chain of filters:
    # 1. highpass → remove low rumble
    # 2. lowpass → remove harsh high freq noise
    # 3. afftdn → noise reduction
    # 4. compand → smooth dynamics (makes speech clearer)
    "highpass=f=80, lowpass=f=12000, afftdn=nf=-22:tn=1, equalizer=f=3000:t=q:w=1:g=3, compand",
    output_path,
    "-y"
])

print(f"Done! Output saved at: {output_path}")

# second we get rid of the interviewer segments of the convo and encode them separately

In [26]:
import pandas as pd
import os

import librosa
import soundfile as sf
import os
import numpy as np
def _normalize_visual_df(df, prefix):
    # Some OpenFace exports use leading spaces in headers.
    df = df.rename(columns={c: c.strip() for c in df.columns}).copy()

    required_keys = ["frame", "timestamp"]
    missing = [c for c in required_keys if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns {missing} in {prefix} file")

    # Drop quality-control columns from every visual dataset.
    drop_cols = [c for c in ["confidence", "success"] if c in df.columns]
    if drop_cols:
        df = df.drop(columns=drop_cols)

    # Prefix all remaining non-join columns to avoid collisions.
    feature_cols = [c for c in df.columns if c not in required_keys]
    rename_map = {c: f"{prefix}_{c}" for c in feature_cols}
    return df.rename(columns=rename_map)

def process_full_multimodal_alignment(p_id, p_folder, target_sr=16000,output_dir="data"):
    # 1. Load transcript
    df_t = pd.read_csv(os.path.join(p_folder, f"{p_id}_TRANSCRIPT.csv"), sep='\t')

    # 2. Load OpenFace streams
    df_clnf = pd.read_csv(os.path.join(p_folder, f"{p_id}_CLNF_features3D.txt"))
    df_gaze = pd.read_csv(os.path.join(p_folder, f"{p_id}_CLNF_gaze.txt"))
    df_pose = pd.read_csv(os.path.join(p_folder, f"{p_id}_CLNF_pose.txt"))
    df_au = pd.read_csv(os.path.join(p_folder, f"{p_id}_CLNF_AUs.txt"))

    # 3. Normalize and disambiguate columns before merging
    df_clnf = _normalize_visual_df(df_clnf, "clnf")
    df_gaze = _normalize_visual_df(df_gaze, "gaze")
    df_pose = _normalize_visual_df(df_pose, "pose")
    df_au = _normalize_visual_df(df_au, "au")

    # Merge visual features on frame+timestamp into one master table.
    df_visual = (
        df_clnf
        .merge(df_gaze, on=["frame", "timestamp"], how="inner")
        .merge(df_pose, on=["frame", "timestamp"], how="inner")
        .merge(df_au, on=["frame", "timestamp"], how="inner")
    )

    processed_turns = []
    last_question = "Initial Greeting"

    # audio processing params
    raw_audio_path = os.path.join(p_folder, f"{p_id}_AUDIO.wav")
    cumulative_time = 0.0
    manifest = []
    # Load the full audio once to save I/O overhead
    # We use sr=None to get original, then resample to 16k for the model
    y, sr = librosa.load(raw_audio_path, sr=target_sr)
    
    stitched_audio = []

    # Handle slight transcript schema variations.
    stop_col = "stop_time" if "stop_time" in df_t.columns else "end_time"

    for _, row in df_t.iterrows():
        text = str(row["value"]).lower()

        # Interviewer turn updates context anchor.
        if row["speaker"] == "Ellie":
            if "?" in text or any(q in text for q in ["how", "why", "tell me", "describe", "do you", "did you", "can you", "when", "have you"]):
                if len(text.split()) > 3:
                    last_question = text
            continue

        # Participant turn gets aligned to visual window.
        if row["speaker"] == "Participant":
            start, stop = row["start_time"], row[stop_col]
            
            visual_mask = (df_visual["timestamp"] >= start) & (df_visual["timestamp"] <= stop)
            turn_visual_data = df_visual[visual_mask]

            duration = stop - start
            new_start = cumulative_time
            new_stop = cumulative_time + duration
            start_sample = int(start * target_sr)
            stop_sample = int(stop * target_sr)
            segment = y[start_sample:stop_sample]
            stitched_audio.extend(segment)

            if not turn_visual_data.empty:
                processed_turns.append({
                    "participant_id": p_id,
                    "question_context": last_question,
                    "answer_text": text,
                    "start_time": start,
                    "stop_time": stop,
                    "visual_features": turn_visual_data.drop(columns=["frame", "timestamp"]).values,
                    'stitched_audio_start': new_start,
                    'stitched_audio_stop': new_stop,
                })
            cumulative_time += duration
            
    output_path = os.path.join(output_dir, f"{p_id}_CLEAN_AUDIO.wav")
    sf.write(output_path, stitched_audio, target_sr)
    print(f"Visual columns: {len(df_visual.columns)}")
    return processed_turns

# Example usage:
p_id = "301"
p_folder = r"/Users/gurusai/Desktop/DAIC_Raw/301_P"
aligned_data = process_full_multimodal_alignment(p_id, p_folder)
print(f"Aligned participant turns: {len(aligned_data)}")

Visual columns: 244
Aligned participant turns: 104


# Encoding features 

ellie -> bert -> text embedding
patient -> ALBERT -> text embedding
audio -> wav2vac -> audio embedding- fit to text size
video features -> GRU -> vid embedding - fit to text size

In [32]:
import torch
import librosa
from transformers import AlbertModel, AlbertTokenizer, Wav2Vec2Model, Wav2Vec2Processor, BertModel, BertTokenizer
from torch import nn
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

# 1. Text: ALBERT (for Participant)
text_tokenizer = AlbertTokenizer.from_pretrained("albert-large-v2")
text_model = AlbertModel.from_pretrained("albert-large-v2").to(device).eval()

# 2. Context: BERT (for Ellie's Questions)
context_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
context_model = BertModel.from_pretrained("bert-base-uncased").to(device).eval()

# 3. Audio: Wav2Vec2
audio_processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
audio_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base").to(device).eval()

# 4. video feature processing 
visual_proj = nn.Linear(244, 768).to(device)

def extract_audio_features(stitched_audio_path, chunk_duration=30):
    """
    Load stitched audio and extract Wav2Vec2 features in chunks to avoid OOM.
    
    Args:
        stitched_audio_path: Path to audio file
        chunk_duration: Process in chunks of N seconds (default 30s)
    
    Returns:
        audio_features: tensor of shape [1, Total_Audio_Frames, 768]
    """
    # Load audio
    y, sr = librosa.load(stitched_audio_path, sr=16000)
    chunk_samples = int(chunk_duration * sr)
    
    all_features = []
    
    # Process in chunks
    for start_idx in range(0, len(y), chunk_samples):
        end_idx = min(start_idx + chunk_samples, len(y))
        chunk = y[start_idx:end_idx]
        
        # Process chunk with Wav2Vec2Processor
        input_values = audio_processor(chunk, return_tensors="pt", sampling_rate=16000).input_values.to(device)
        
        # Extract features with model
        with torch.no_grad():
            chunk_features = audio_model(input_values).last_hidden_state  # [1, chunk_frames, 768]
        
        all_features.append(chunk_features)
    
    # Concatenate all chunks along time dimension
    audio_features = torch.cat(all_features, dim=1)  # [1, Total_Audio_Frames, 768]
    
    return audio_features


# Example usage:
stitched_audio_path = r'data/301_CLEAN_AUDIO.wav'  # from previous cell
audio_features = extract_audio_features(stitched_audio_path, chunk_duration=30)
print(f"Audio features shape: {audio_features.shape}")


Device: mps


Loading weights: 100%|██████████| 25/25 [00:00<00:00, 11331.06it/s]
AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7793.56it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.tr

Audio features shape: torch.Size([1, 23756, 768])


In [ ]:
import torch.nn.functional as F

TARGET_LENGTH = 128  # Target audio sequence length after pooling


def build_multimodal_packets(aligned_data, audio_features,total_audio_samples, target_sr=16000):
    """
    Iterate through aligned_data and build multimodal packets.
    
    For each turn:
    - Encode participant answer with ALBERT (full sequence)
    - Encode interviewer context with BERT (take [CLS] token only)
    - Slice audio features and pool to target length
    
    Returns:
        packets: list of dicts with keys: answer_embed, context_embed, audio_embed, visual_features
    """
    packets = []
    total_feature_frames = audio_features.shape[1]
    samples_per_frame = total_audio_samples / total_feature_frames
    
    for turn in aligned_data:
        # ============= A. TEXT & CONTEXT ENCODING =============
        
        # 1. Participant Answer → ALBERT (full sequence)
        answer_text = turn["answer_text"]
        answer_tokens = text_tokenizer(
            answer_text,
            truncation=True,
            padding='max_length',
            max_length=128,
            return_tensors='pt'
        )
        answer_ids = answer_tokens['input_ids'].to(device)
        answer_mask = answer_tokens['attention_mask'].to(device)
        
        with torch.no_grad():
            answer_embed = text_model(answer_ids, answer_mask).last_hidden_state  # [1, 128, 768]
        
        # 2. Interviewer Context → BERT ([CLS] token only)
        context_text = turn["question_context"]
        context_tokens = context_tokenizer(
            context_text,
            truncation=True,
            padding='max_length',
            max_length=128,
            return_tensors='pt'
        )
        context_ids = context_tokens['input_ids'].to(device)
        context_mask = context_tokens['attention_mask'].to(device)
        
        with torch.no_grad():
            context_output = context_model(context_ids, context_mask).last_hidden_state  # [1, 128, 768]
            context_embed = context_output[:, 0, :]  # [CLS] token only → [1, 768]
        
    # ============= B. AUDIO SLICING & POOLING =============
        start_frame = int(round((turn["stitched_audio_start"] * target_sr) / samples_per_frame))
        stop_frame = int(round((turn["stitched_audio_stop"] * target_sr) / samples_per_frame))

        # Guard against zero-length slices
        if stop_frame <= start_frame:
            stop_frame = start_frame + 1
        
        stop_frame = min(stop_frame, total_feature_frames)
        turn_audio = audio_features[:, start_frame:stop_frame, :]
        
        # MPS doesn't support adaptive_avg_pool1d with non-divisible sizes, so move to CPU for pooling
        turn_audio_cpu = turn_audio.cpu()
        turn_audio_t = turn_audio_cpu.transpose(1, 2)
        audio_pooled = F.adaptive_avg_pool1d(turn_audio_t, TARGET_LENGTH)
        audio_embed = audio_pooled.transpose(1, 2)
        

        # ============= C. VISUAL FEATURES =============
        # Guard against zero-length visual data
        if len(turn['visual_features']) == 0:
            # Create a zero-filled dummy if the segment has no visual frames
            vis_embed = torch.zeros((1, TARGET_LENGTH, 768))
        else:
            raw_visual = torch.from_numpy(turn['visual_features']).float().to(device)
            with torch.no_grad(): # Crucial: don't track gradients during extraction
                vis_projected = visual_proj(raw_visual).unsqueeze(0) 
                # Move to CPU for pooling (MPS limitation)
                vis_projected_cpu = vis_projected.cpu()
                vis_pooled = F.adaptive_avg_pool1d(vis_projected_cpu.transpose(1, 2), TARGET_LENGTH)
                vis_embed = vis_pooled.transpose(1, 2)
        # ============= COLLECT MULTIMODAL PACKET =============
        
        packet = {
            'answer_embed': answer_embed.squeeze(0).cpu(), # Move to CPU to save VRAM
            'context_embed': context_embed.squeeze(0).cpu(),
            'audio_embed': audio_embed.squeeze(0),  # Already on CPU
            'visual_embed': vis_embed.squeeze(0), # Already on CPU
            'mask': answer_mask.squeeze(0).cpu() # [128]
        }
        
        packets.append(packet)
        
        print(f"Turn {len(packets)}: answer={answer_embed.shape}, context={context_embed.shape}, audio={audio_embed.shape}")
    
    return packets


# Example usage:
stitched_audio_path = f"data/{p_id}_CLEAN_AUDIO.wav"
audio_features = extract_audio_features(stitched_audio_path)
print(f"Audio features shape: {audio_features.shape}")
#
packets = build_multimodal_packets(aligned_data, audio_features, total_audio_samples=len(librosa.load(stitched_audio_path, sr=16000)[0]))
print(f"Built {len(packets)} multimodal packets")


Audio features shape: torch.Size([1, 23756, 768])


RuntimeError: Adaptive pool MPS: input sizes must be divisible by output sizes. Non-divisible input sizes are not implemented on MPS device yet. For now, you can manually transfer tensor to cpu in this case. Please refer to [this issue](https://github.com/pytorch/pytorch/issues/96056)